## Exercise 1: Conceptual Questions

### 1. Q, K, V Analogy (Library Retrieval System)

In a library retrieval system, the vectors represent the following roles:

* **Query ($Q$):** This represents the **search term or topic** you type into the library catalog (e.g., *"How do transformers work?"*). In the model, it's the current word looking for context.
* **Key ($K$):** This represents the **catalog card or book index attributes** (e.g., titles, tags, or summaries of all books in the library). The system matches your query against these keys to see which books are relevant.
* **Value ($V$):** This represents the **actual content of the books** themselves. Once you find the best-matching keys, you extract the information from these corresponding values.

### 2. The Scaling Factor ($\frac{1}{\sqrt{d_k}}$)

If the embedding dimension $d_k$ is large, the dot products grow significantly in magnitude. When you pass very large numbers into the softmax function, the output distribution becomes extremely peaked, pushing the highest score close to $1.0$ and others to $0.0$.

**Why this breaks training:**

This causes the softmax function to enter its "saturation regions," where its curve is almost completely flat. Mathematically, the derivative (gradient) of softmax in these regions approaches zero. This leads to the **vanishing gradient problem**, effectively halting backpropagation because weights can no longer update. Scaling down by $\frac{1}{\sqrt{d_k}}$ keeps the variance of the dot products at $1$, ensuring healthy gradients.

### 3. Self-Attention vs. Cross-Attention

The cross-attention layer acts as the bridge between the input language (Encoder) and the target language (Decoder).

* **The Purpose:** It allows the decoder to look back at the entire input sequence while it generates the translation word-by-word.
* **How it works:** By using Decoder vectors for $Q$ and Encoder vectors for $K$ and $V$, the decoder asks: *"Given the word I just generated ($Q$), which parts of the original source text ($K$) should I focus on to extract the next piece of meaning ($V$)?"*


## Exercise 2 & 3: Code Implementation

In [1]:
# Cell 1: Imports and Function Definition
import torch
import torch.nn.functional as F

def scaled_dot_product_attention(q, k, v, mask=None):
    """
    Computes Scaled Dot-Product Attention.
    
    Args:
        q: Query tensor of shape (batch_size, seq_len, d_k)
        k: Key tensor of shape (batch_size, seq_len, d_k)
        v: Value tensor of shape (batch_size, seq_len, d_k)
        mask: Optional boolean tensor of shape (seq_len, seq_len) or broadcastable. 
              True indicates positions to mask out.
    """
    # Step 1: Dot Product Scores
    scores = torch.matmul(q, k.transpose(-2, -1))
    print(f"Step 1 - Scores Shape: {scores.shape}")
    
    # Step 2: Scaling
    d_k = k.size(-1)
    scaled_scores = scores / torch.sqrt(torch.tensor(d_k, dtype=torch.float32))
    
    # Exercise 3: Masking Application
    if mask is not None:
        # Softmax converts values to probabilities using exponentiation.
        # Very negative values become approximately zero after softmax.
        scaled_scores = scaled_scores.masked_fill(mask, -1e9)
        
    # Step 3: Softmax to get Attention Weights
    attention_weights = F.softmax(scaled_scores, dim=-1)
    print("\nStep 3 - Attention Weights Distribution:")
    print(attention_weights)
    
    # Step 4: Weighted Sum of Values
    output = torch.matmul(attention_weights, v)
    print(f"\nStep 4 - Output Shape: {output.shape}")
    
    return output, attention_weights


In [2]:
# Cell 2: Testing without a Mask (Exercise 2)
torch.manual_seed(42)  # For reproducible random tensors

# Setup dimensions: (batch_size=1, seq_len=4, d_k=8)
queries = torch.randn(1, 4, 8)
keys = torch.randn(1, 4, 8)
values = torch.randn(1, 4, 8)

print("=== RUNNING EXERCISE 2 (UNMASKED) ===")
output, weights = scaled_dot_product_attention(queries, keys, values)


=== RUNNING EXERCISE 2 (UNMASKED) ===
Step 1 - Scores Shape: torch.Size([1, 4, 4])

Step 3 - Attention Weights Distribution:
tensor([[[0.1942, 0.4102, 0.3242, 0.0714],
         [0.0408, 0.6555, 0.0864, 0.2173],
         [0.1041, 0.2057, 0.1824, 0.5078],
         [0.1926, 0.1803, 0.1955, 0.4316]]])

Step 4 - Output Shape: torch.Size([1, 4, 8])


In [3]:
# Cell 3: Testing with causal upper-triangular mask (Exercise 3)
print("\n=== RUNNING EXERCISE 3 (MASKED DECODING) ===")

seq_len = 4

# Creates an upper triangular matrix of True values
# representing future tokens to hide.
causal_mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1).bool()

print("Causal Mask Tensor (True means hide/mask):")
print(causal_mask)

output_masked, weights_masked = scaled_dot_product_attention(
    queries, keys, values, mask=causal_mask
)



=== RUNNING EXERCISE 3 (MASKED DECODING) ===
Causal Mask Tensor (True means hide/mask):
tensor([[False,  True,  True,  True],
        [False, False,  True,  True],
        [False, False, False,  True],
        [False, False, False, False]])
Step 1 - Scores Shape: torch.Size([1, 4, 4])

Step 3 - Attention Weights Distribution:
tensor([[[1.0000, 0.0000, 0.0000, 0.0000],
         [0.0585, 0.9415, 0.0000, 0.0000],
         [0.2114, 0.4180, 0.3706, 0.0000],
         [0.1926, 0.1803, 0.1955, 0.4316]]])

Step 4 - Output Shape: torch.Size([1, 4, 8])
